In [ ]:
import os

print("Does the folder exist?", os.path.exists('/kaggle/input/plantvillage-dataset'))
print()
print("Contents of /kaggle/input:")
print(os.listdir('/kaggle/input'))

In [ ]:
import os

print("Contents of /kaggle/input/datasets:")
print(os.listdir('/kaggle/input/datasets'))

In [ ]:
import os

print("Inside abdallahalidev:")
print(os.listdir('/kaggle/input/datasets/abdallahalidev'))

In [ ]:
import os

print("Inside plantvillage-dataset:")
print(os.listdir('/kaggle/input/datasets/abdallahalidev/plantvillage-dataset'))

In [ ]:
import os

color_path = '/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color'
print("Inside color/:")
contents = os.listdir(color_path)
print(f"Number of items: {len(contents)}")
print("Sample items:", contents[:5])

# Cell 1_Importing libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import confusion_matrix, classification_report

print("Tensorflow Version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


#  Cell 2_Finding the actual image folder and splitting the dataset into train/val/test

In [ ]:
import os
import shutil
import random

source_dir = '/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color'
output_dir = '/kaggle/working/plant_split'  # writable directory on Kaggle

classes = os.listdir(source_dir)
print(f"Found {len(classes)} classes")
print("Sample classes:", classes[:5])

for split in ['train', 'val', 'test']:
    for cls in classes:
        os.makedirs(os.path.join(output_dir, split, cls), exist_ok=True)

def split_images(class_name, train_pct=0.8, val_pct=0.1):
    folder = os.path.join(source_dir, class_name)
    images = os.listdir(folder)
    random.shuffle(images)

    total = len(images)
    train_end = int(total * train_pct)
    val_end = int(total * (train_pct + val_pct))

    splits = {
        'train': images[:train_end],
        'val': images[train_end:val_end],
        'test': images[val_end:]
    }

    for split_name, files in splits.items():
        dest = os.path.join(output_dir, split_name, class_name)
        for f in files:
            shutil.copy(os.path.join(folder, f), os.path.join(dest, f))

for cls in classes:
    split_images(cls)

print("Split complete!")
print("Train classes:", len(os.listdir(f'{output_dir}/train')))

# Step 3 — Data generators

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

train_dir = '/kaggle/working/plant_split/train'
val_dir   = '/kaggle/working/plant_split/val'
test_dir  = '/kaggle/working/plant_split/test'

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import preprocess_input

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    horizontal_flip=True,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1
)

val_test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_gen = train_datagen.flow_from_directory(
    train_dir, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical'
)

val_gen = val_test_datagen.flow_from_directory(
    val_dir, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical'
)

test_gen = val_test_datagen.flow_from_directory(
    test_dir, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)

num_classes = len(train_gen.class_indices)
print(f"Classes: {num_classes}")

# Cell 4_Building the model with ResNet50(frozen base)

In [ ]:
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
x=base_model.output
x=GlobalAveragePooling2D()(x)
x=Dense(256, activation='relu')(x)
x=BatchNormalization()(x)
x=Dropout(0.4)(x)
predictions = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


# Cell 5_Phase 1 of training (frozen base) __ getting a base line 

In [ ]:
callbacks_phase1 = [
    EarlyStopping(patience=4, restore_best_weights=True),
    ReduceLROnPlateau(factor=0.5, patience=2, verbose=1)
]

history_frozen = model.fit(
    train_gen,
    epochs=10,
    validation_data=val_gen,
    callbacks=callbacks_phase1
)

baseline_loss, baseline_acc = model.evaluate(val_gen)
print(f"\nFrozen-base validation accuracy: {baseline_acc*100:.2f}%")

# Cell 6_Phase 2: unfreezing and fine-tuning

In [ ]:
# Unfreeze the last 30 layers of ResNet50
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

# Recompile with a much lower learning rate — critical for fine-tuning
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_phase2 = [
    EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(factor=0.5, patience=3, verbose=1),
    ModelCheckpoint('/content/best_plant_model.h5', save_best_only=True, verbose=1)
]

history_finetuned = model.fit(
    train_gen,
    epochs=20,
    validation_data=val_gen,
    callbacks=callbacks_phase2
)

# Cell 7_ Comparing frozen vs fine-tuned & saving plots

In [ ]:
acc_frozen = history_frozen.history['val_accuracy']
acc_finetuned = history_finetuned.history['val_accuracy']
all_acc = acc_frozen + acc_finetuned

loss_frozen = history_frozen.history['val_loss']
loss_finetuned = history_finetuned.history['val_loss']
all_loss = loss_frozen + loss_finetuned

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(all_acc)
ax1.axvline(x=len(acc_frozen)-1, color='r', linestyle='--', label='Fine-tuning-starts')
ax2.set_title('Validation loss')
ax2.set_xlabel('Epochs')
ax2.legend()

plt.tight_layout()
plt.savefig('/kaggle/working/accuracy_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Frozen-base best val accuracy:    {max(acc_frozen)*100:.2f}%")
print(f"Fine-tuned best val accuracy:    {max(acc_finetuned)*100:.2f}%")
print(f"Improvement from fine-tuning:    {(max(acc_finetuned)-max(acc_frozen))*100:.2f}%")

# Cell 8_Evaluating on the test set + confusion matrix

In [ ]:
test_loss, test_acc = model.evaluate(test_gen)
print(f"\nFinal test accuracy: {test_acc*100:.2f}%")

y_pred = model.predict(test_gen)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = test_gen.classes

class_names = list(test_gen.class_indices.keys())

# Confusion matrix (too many classes for a full heatmap — show top confused pairs)
cm = confusion_matrix(y_true, y_pred_classes)

# Show only the 15 most frequent classes to keep it readable
top_classes_idx = np.argsort(-cm.sum(axis=1))[:15]
cm_subset = cm[np.ix_(top_classes_idx, top_classes_idx)]
names_subset = [class_names[i] for i in top_classes_idx]

plt.figure(figsize=(12, 10))
sns.heatmap(cm_subset, annot=True, fmt='d', cmap='Blues',
            xticklabels=names_subset, yticklabels=names_subset)
plt.title('Confusion matrix (top 15 most frequent classes)')
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nClassification report:")
print(classification_report(y_true, y_pred_classes, target_names=class_names, zero_division=0))

# Cell 9 — Saving the model and class names


In [ ]:
model.save('/kaggle/working/plant_disease_model.h5')

import json
with open('/kaggle/working/class_names.json', 'w') as f:
    json.dump(class_names, f)

print("Model and class names saved!")

# Cell 10_ Building the Gradio demo

In [ ]:
!pip install gradio -q

import gradio as gr
from tensorflow.keras.preprocessing import image as keras_image

def predict_disease(img):
    img = img.resize((IMG_SIZE, IMG_SIZE))
    img_array = keras_image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)

    predictions = model.predict(img_array)[0]
    top_5_idx = predictions.argsort()[-5:][::-1]

    results = {class_names[i]: float(predictions[i]) for i in top_5_idx}
    return results

demo = gr.Interface(
    fn=predict_disease,
    inputs=gr.Image(type="pil", label="Upload a leaf photo"),
    outputs=gr.Label(num_top_classes=5, label="Diagnosis"),
    title="Plant Disease Detector",
    description="Upload a photo of a plant leaf to detect disease using a fine-tuned ResNet50 model.",
    examples=None
)

demo.launch(share=True)

# Cell 11_ Pushing everything to GitHub